In [ ]:
from __future__ import annotations

import argparse
import json
import os
import tempfile
from dataclasses import asdict, dataclass
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pywt
from skimage.color import rgb2gray
from skimage.feature import graycomatrix, graycoprops
from skimage.io import imread
from skimage.transform import resize
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


LABELS = {"good": 0, "not-good": 1}
CLASS_NAMES = ["good", "not-good"]

In [ ]:
@dataclass(frozen=True)
class FeatureConfig:
    image_size: int = 256 # Resize images to image-size x image-size before feature extraction. This standardization is important for consistent feature extraction, especially for GLCM and DWT, which can be sensitive to image dimensions. A size of 256x256 provides a good balance between retaining detail and keeping computational requirements manageable.
    glcm_levels: int = 32 # The number of gray levels to quantize the image into for GLCM calculation. Reducing to 32 levels helps manage the size of the co-occurrence matrix and focuses on broader texture patterns, which can be beneficial for classification while keeping computational complexity reasonable. TODO validate that with paper
    glcm_distances: tuple[int, ...] = (1, 2, 4, 8)
    glcm_angles: tuple[float, ...] = (0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4)
    glcm_properties: tuple[str, ...] = (
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM",
    )
    dwt_wavelet: str = "db2" #?
    dwt_level: int = 3
    dataset_dir: Path = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
    output_model: Path = Path("steelblast_svm_glcm_dwt.joblib")
    metrics_json: Path = Path("steelblast_svm_glcm_dwt_metrics.json")
    heatmap_dir: Path = Path("steelblast_svm_glcm_dwt_focus_heatmaps")
    occlusion_patch_size: int = 32
    occlusion_stride: int = 16
    image_size: int = 256
    quick_limit: int | None = None
    # Parallel jobs for GridSearchCV. Use -1 outside restricted environments.
    n_jobs: int = 1

# load either train or test images 
def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)
    




config = FeatureConfig()

train_paths, y_train = load_split(config.dataset_dir, "train")
test_paths, y_test = load_split(config.dataset_dir, "test")

In [ ]:
# Loops through all images
# Extracts features
# Stacks them into matrix:
def build_feature_matrix(
    image_paths: list[Path],
    config: FeatureConfig,
    split_name: str,
) -> np.ndarray:
    features = []

    for index, image_path in enumerate(image_paths, start=1):
        features.append(extract_features(image_path, config))
        if index % 100 == 0 or index == len(image_paths):
            print(f"{split_name}: extracted {index}/{len(image_paths)} images")

    return np.vstack(features)

def balanced_limit(
    image_paths: list[Path],
    labels: np.ndarray,
    limit_per_class: int,
) -> tuple[list[Path], np.ndarray]:
    selected_paths: list[Path] = []
    selected_labels: list[int] = []

    for label in sorted(np.unique(labels)):
        class_indices = np.flatnonzero(labels == label)[:limit_per_class]
        selected_paths.extend(image_paths[index] for index in class_indices)
        selected_labels.extend(labels[index] for index in class_indices)

    return selected_paths, np.asarray(selected_labels, dtype=np.int64)

def extract_features(image_path: Path, config: FeatureConfig) -> np.ndarray:
    image = read_grayscale_image(image_path, config.image_size)
    return extract_features_from_image(image, config)

# TODO NN won't use grayscale
def read_grayscale_image(image_path: Path, image_size: int) -> np.ndarray:
    image = imread(image_path)

    if image.ndim == 3:
        image = image[..., :3]
        image = rgb2gray(image) # Convert to grayscale using luminosity method, which accounts for human perception of color brightness. This is important for texture analysis with GLCM, as it relies on intensity values. The resulting grayscale image will have values in the range [0, 1], which is suitable for further processing and feature extraction.

    image = resize(
        image,
        (image_size, image_size),
        anti_aliasing=True,
        preserve_range=True,
    )

    image = image.astype(np.float32)
    if image.max() > 1.0:
        image /= 255.0

    return image

# get Final feature vector
def extract_features_from_image(image: np.ndarray, config: FeatureConfig) -> np.ndarray:
    glcm_features = extract_glcm_features(image, config)
    dwt_features = extract_dwt_features(image, config)
    return np.concatenate([glcm_features, dwt_features])

def extract_glcm_features(image: np.ndarray, config: FeatureConfig) -> np.ndarray:
    #reduce to 32 intensity levels
    quantized = quantize_image(image, config.glcm_levels)
    # Co-occurrence Matrix
    glcm = graycomatrix(
        quantized,
        distances=config.glcm_distances,
        angles=config.glcm_angles,
        levels=config.glcm_levels,
        symmetric=True,
        normed=True,
    )

    features: list[float] = []
    # extract 6 configured features
    for prop in config.glcm_properties:
        values = graycoprops(glcm, prop).ravel()
        features.extend(values)
        features.append(float(values.mean()))
        features.append(float(values.std()))

    return np.asarray(features, dtype=np.float32)

def extract_dwt_features(image: np.ndarray, config: FeatureConfig) -> np.ndarray:
    # Apply wavelet decomposition
    coeffs = pywt.wavedec2(
        image,
        wavelet=config.dwt_wavelet,
        level=config.dwt_level,
        mode="symmetric",
    )

    features: list[float] = []
    approximation = coeffs[0]
    features.extend(describe_coefficients(approximation))

    # Repeat across 3 levels
    for horizontal, vertical, diagonal in coeffs[1:]:
        features.extend(describe_coefficients(horizontal))
        features.extend(describe_coefficients(vertical))
        features.extend(describe_coefficients(diagonal))

    return np.asarray(features, dtype=np.float32)

def quantize_image(image: np.ndarray, levels: int) -> np.ndarray:
    clipped = np.clip(image, 0.0, 1.0)
    quantized = np.floor(clipped * (levels - 1)).astype(np.uint8)
    return quantized

# DWT coefficients can have a wide range of values, including negative and positive numbers. To capture the texture information effectively, we compute several statistics on the coefficients, such as mean, standard deviation, energy (mean of squared values), and entropy (which measures the randomness or complexity of the coefficients). Additionally, we include percentiles to capture the distribution of coefficient values. These features help summarize the texture information contained in the DWT coefficients in a way that is useful for classification.
def describe_coefficients(coefficients: np.ndarray) -> list[float]:
    values = coefficients.ravel().astype(np.float64)
    abs_values = np.abs(values)
    energy = np.mean(values**2)
    histogram, _ = np.histogram(abs_values, bins=32, density=False)
    probabilities = histogram.astype(np.float64) / max(histogram.sum(), 1)
    probabilities = probabilities[probabilities > 0]
    entropy = -np.sum(probabilities * np.log2(probabilities))

    return [
        float(values.mean()),
        float(values.std()),
        float(abs_values.mean()),
        float(abs_values.std()),
        float(energy),
        float(entropy),
        float(np.percentile(values, 10)),
        float(np.percentile(values, 50)),
        float(np.percentile(values, 90)),
    ]


# for a quick training
if config.quick_limit is not None:
    train_paths, y_train = balanced_limit(train_paths, y_train, config.quick_limit)
    test_paths, y_test = balanced_limit(test_paths, y_test, config.quick_limit)

print(f"Train images: {len(train_paths)}")
print(f"Test images:  {len(test_paths)}")
print(f"Feature config: {config}")

X_train = build_feature_matrix(train_paths, config, "train")
X_test = build_feature_matrix(test_paths, config, "test")


In [ ]:
MODEL_PATH = "steelblast_svm_glcm_dwt.joblib"

# train model using SVM (RBF kernel)
def train_model(X_train: np.ndarray, y_train: np.ndarray, n_jobs: int) -> GridSearchCV:
    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()), # StandardScaler standardizes features by removing the mean and scaling to unit variance, which is important for SVM performance since it relies on distances in feature space. This ensures that all features contribute equally to the model and prevents features with larger scales from dominating the decision boundary.
            ("svm", SVC(kernel="rbf", class_weight="balanced")), # The SVC with RBF kernel is a powerful non-linear classifier that can capture complex relationships in the data. The class_weight="balanced" option automatically adjusts weights inversely proportional to class frequencies, which helps address any class imbalance in the dataset and can improve performance on the minority class.
        ]
    )
    # Define the hyperparameter grid for GridSearchCV
    param_grid = {
        "svm__C": [0.1, 1, 10, 100], # The C parameter controls the trade-off between achieving a low training error and a low testing error (generalization). A smaller C encourages a simpler decision boundary that may misclassify some training points but generalizes better, while a larger C tries to classify all training points correctly, which can lead to overfitting. Testing a range of values allows us to find the optimal balance for our dataset.
        "svm__gamma": ["scale", 0.01, 0.001, 0.0001], # The gamma parameter defines how far the influence of a single training example reaches. A low gamma means that the model considers points at a larger distance from the decision boundary, resulting in a smoother decision surface. A high gamma focuses more on points close to the decision boundary, which can lead to a more complex model that may overfit. Including "scale" allows the model to automatically adjust gamma based on the number of features, which can be a good default choice.
    }

    min_class_count = int(np.bincount(y_train).min())
    #at most 5 folds
    n_splits = min(5, min_class_count)
    if n_splits < 2:
        raise ValueError("Each class needs at least two training images.")
    #Stratified splitting keeps the same class proportions in every fold as in the whole dataset.
    #random_state ensures reproducibility, and shuffle=True randomizes the data before splitting to avoid any order bias.
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    #CV cross validation
    #GridSearchCV exhaustively tries all combinations of the specified hyperparameters (C and gamma for the SVM) and evaluates each combination using cross-validation on the training data. It selects the combination that yields the best average F1 score across the folds.
    search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=n_jobs,
        verbose=2,
    )
    search.fit(X_train, y_train)
    return search


if os.path.exists(MODEL_PATH):
    print("Loading existing model...")
    model = joblib.load(MODEL_PATH)
else:

    print(f"Training feature matrix: {X_train.shape}")
    print(f"Testing feature matrix:  {X_test.shape}")

    model = train_model(X_train, y_train, config.n_jobs)

y_pred = model.predict(X_test)

report = classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
matrix = confusion_matrix(y_test, y_pred).tolist()

print(f"Best parameters: {model.best_params_}")
print(f"Best CV F1: {model.best_score_:.4f}")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
print("Confusion matrix:")
print(np.asarray(matrix))

joblib.dump(model, MODEL_PATH)


In [ ]:
# Computes confidence score for each prediction
def predicted_label_confidences(model: Pipeline, features: np.ndarray) -> np.ndarray:
    predictions = model.predict(features)
    # decision_function gives distance to the separating hyperplane. For binary classification, it's a single score where positive means class 1 and negative means class 0. Magnitude of distance = confidence
    scores = model.decision_function(features)
    # 
    if scores.ndim == 1:
        positive_class = int(model.classes_[1])
        signed_scores = np.where(predictions == positive_class, scores, -scores)
        return signed_scores.astype(np.float64)

    return np.asarray(
        [scores[index, np.flatnonzero(model.classes_ == label)[0]] for index, label in enumerate(predictions)],
        dtype=np.float64,
    )

def prediction_confidence(model: Pipeline, features: np.ndarray, label: int) -> float:
    scores = model.decision_function(features.reshape(1, -1))

    if np.ndim(scores) == 1:
        positive_class = int(model.classes_[1])
        score = float(scores[0])
        return score if label == positive_class else -score

    class_index = int(np.flatnonzero(model.classes_ == label)[0])
    return float(scores[0, class_index])

def read_display_image(image_path: Path, image_size: int) -> np.ndarray:
    image = imread(image_path)

    if image.ndim == 2:
        image = np.repeat(image[..., np.newaxis], 3, axis=2)
    else:
        image = image[..., :3]

    image = resize(
        image,
        (image_size, image_size),
        anti_aliasing=True,
        preserve_range=True,
    )

    image = image.astype(np.float32)
    if image.max() > 1.0:
        image /= 255.0

    return np.clip(image, 0.0, 1.0)

def save_focus_pair(
    display_image: np.ndarray,
    heatmap: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), constrained_layout=True)
    axes[0].imshow(display_image)
    axes[0].set_title("Example image")
    axes[0].set_axis_off()

    axes[1].imshow(display_image)
    vmax = float(heatmap.max()) if heatmap.size and heatmap.max() > 0 else 1.0
    overlay = axes[1].imshow(heatmap, cmap="jet", alpha=0.62, vmin=0, vmax=vmax)
    axes[1].set_title("Activation heatmap")
    axes[1].set_axis_off()
    fig.colorbar(overlay, ax=axes[1], label="Decision-score drop")
    fig.suptitle(title, fontsize=16)
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


# Mask parts of the image and see how prediction confidence changes.
# Steps:
# 1. Compute base prediction confidence
# 2. Slide a patch over the image
# 3. Replace patch with median value
# 4. Recompute prediction
# 5. Measure confidence drop
def compute_occlusion_focus_heatmap(
    image: np.ndarray,
    model: Pipeline,
    predicted_label: int,
    config: FeatureConfig,
    patch_size: int, # The size of the square region (in pixels) that gets “hidden” at each step.
    stride: int, # The number of pixels the patch moves horizontally and vertically between steps. Smaller strides create smoother heatmaps but require more computations.
) -> np.ndarray:
    base_features = extract_features_from_image(image, config)
    base_confidence = prediction_confidence(model, base_features, predicted_label)
    heatmap = np.zeros_like(image, dtype=np.float32)
    counts = np.zeros_like(image, dtype=np.float32)
    fill_value = float(np.median(image))
    height, width = image.shape

    for top in range(0, height, stride):
        bottom = min(top + patch_size, height)
        top = max(0, bottom - patch_size)

        for left in range(0, width, stride):
            right = min(left + patch_size, width)
            left = max(0, right - patch_size)

            occluded = image.copy()
            occluded[top:bottom, left:right] = fill_value
            occluded_features = extract_features_from_image(occluded, config)
            occluded_confidence = prediction_confidence(model, occluded_features, predicted_label)
            importance = max(0.0, base_confidence - occluded_confidence)

            heatmap[top:bottom, left:right] += importance
            counts[top:bottom, left:right] += 1

    return np.divide(heatmap, counts, out=np.zeros_like(heatmap), where=counts > 0)

import shutil

def reset_directory(output_dir: Path):
    if output_dir.exists():
        shutil.rmtree(output_dir)  # ❌ delete everything inside
    output_dir.mkdir(parents=True, exist_ok=True)  # ✅ recreate empty folder

from pathlib import Path
import numpy as np
from typing import Dict, List, Any

def save_classification_case_heatmaps(
    test_paths: list[Path],
    y_test: np.ndarray,
    y_pred: np.ndarray,
    X_test: np.ndarray,
    model: Pipeline,
    config: FeatureConfig,
    output_dir: Path,
    patch_size: int,
    stride: int,
    max_per_case: int | None = None,   # ✅ optional limit
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Generate and save occlusion heatmaps for ALL classification cases (TP, FP, FN, TN).

    Returns:
        Dictionary mapping case -> list of metadata dicts.
    """

    reset_directory(output_dir)

    heatmap_results: Dict[str, List[Dict[str, Any]]] = {}
    confidences = predicted_label_confidences(model, X_test)

    case_definitions = {
        "true_positive": (1, 1),
        "false_positive": (0, 1),
        "false_negative": (1, 0),
        "true_negative": (0, 0),
    }

    for case_name, (actual_label, predicted_label) in case_definitions.items():

        # ✅ Collect indices
        case_indices = [
            i for i, (a, p) in enumerate(zip(y_test, y_pred))
            if a == actual_label and p == predicted_label
        ]

        # ✅ Sort by confidence (highest first)
        case_indices = sorted(
            case_indices,
            key=lambda i: confidences[i],
            reverse=True,
        )

        # ✅ Optional limit (useful for large datasets)
        if max_per_case is not None:
            case_indices = case_indices[:max_per_case]

        case_dir = output_dir / case_name
        case_dir.mkdir(parents=True, exist_ok=True)

        heatmap_results[case_name] = []

        print(f"\nProcessing {case_name}: {len(case_indices)} images")

        for rank, idx in enumerate(case_indices):

            try:
                image_path = test_paths[idx]

                # ✅ Load images
                grayscale_image = read_grayscale_image(
                    image_path, config.image_size
                )
                display_image = read_display_image(
                    image_path, config.image_size
                )

                # ✅ Compute heatmap
                heatmap = compute_occlusion_focus_heatmap(
                    grayscale_image,
                    model,
                    int(y_pred[idx]),
                    config,
                    patch_size,
                    stride,
                )

                # ✅ Create unique filename
                output_path = case_dir / (
                    f"{rank:03d}_{image_path.stem}_heatmap.png"
                )

                title = (
                    f"{case_name.replace('_', ' ').title()} | "
                    f"actual {CLASS_NAMES[int(y_test[idx])]}, "
                    f"predicted {CLASS_NAMES[int(y_pred[idx])]} | "
                    f"conf {confidences[idx]:.3f}"
                )

                # ✅ Save visualization
                save_focus_pair(display_image, heatmap, title, output_path)

                # ✅ Store metadata
                result = {
                    "index": int(idx),
                    "rank": rank,
                    "actual_label": CLASS_NAMES[int(y_test[idx])],
                    "predicted_label": CLASS_NAMES[int(y_pred[idx])],
                    "source_image": str(image_path),
                    "heatmap": str(output_path),
                    "confidence": float(confidences[idx]),
                    "max_activation": float(np.max(heatmap)),
                }

                heatmap_results[case_name].append(result)

            except Exception as e:
                print(f"⚠️ Failed on index {idx}: {e}")

        print(f"✅ Done {case_name}: saved {len(heatmap_results[case_name])}")

    return heatmap_results

heatmap_paths = save_classification_case_heatmaps(
    test_paths,
    y_test,
    y_pred,
    X_test,
    model.best_estimator_,
    config,
    config.heatmap_dir,
    config.occlusion_patch_size,
    config.occlusion_stride,
    max_per_case=3
)


metrics = {
    "best_params": model.best_params_,
    "best_cv_f1": float(model.best_score_),
    "classification_report": report,
    "confusion_matrix": matrix,
    "train_images": len(train_paths),
    "test_images": len(test_paths),
    "feature_count": int(X_train.shape[1]),
    "heatmap_method": "occlusion_sensitivity",
    "heatmap_settings": {
        "heatmaps_per_case": 1,
        "occlusion_patch_size": config.occlusion_patch_size,
        "occlusion_stride": config.occlusion_stride,
    },
    "heatmaps": heatmap_paths,
}
config.metrics_json.write_text(json.dumps(metrics, indent=2))

print(f"Saved model to:   {config.output_model.resolve()}")
print(f"Saved metrics to: {config.metrics_json.resolve()}")
print(f"Saved heatmaps to: {config.heatmap_dir.resolve()}")
